In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [10]:
from pathlib import Path

root = Path('/content/drive/MyDrive')
print('My Drive available:', root.exists())

for item in sorted(root.iterdir()):
    print(item.name)

print('\nZIP files:')
for item in root.rglob('*'):
    if item.is_file() and item.suffix.lower() == '.zip':
        print(item)

My Drive available: True
123158859.pdf
1716129454954.jpg
1784967315 - 25000.00 To HARCOURT BUTLER TECHNICAL UNIVERSITY on Google Pay (1).png
1784967315 - 25000.00 To HARCOURT BUTLER TECHNICAL UNIVERSITY on Google Pay (2).png
1784967315 - 25000.00 To HARCOURT BUTLER TECHNICAL UNIVERSITY on Google Pay.png
2023071726.pdf
20250411_003505-COLLAGE.jpg
20250411_003648-COLLAGE.jpg
20250411_003818-COLLAGE.jpg
230108047_Prateekchaurasiya_IIIrd_IT (2).zip
406554_cert (1).pdf
406554_cert.pdf asus warenty .pdf
406554_receipt.pdf asus invoice .pdf
5U7A1796-1.JPG
AUD-20221005-WA0002 (1).mp3
AUD-20221005-WA0002.mp3
AUD-20221005-WA0003.mp3
AUD-20221005-WA0004.mp3
AUD-20221005-WA0005 (1).mp3
AUD-20221005-WA0005 (2).mp3
AUD-20221005-WA0005.mp3
AUD-20221005-WA0006.mp3
AUD-20221101-WA0004 (1).opus
AUD-20221101-WA0004 (2).opus
AUD-20221101-WA0004.opus
AdmitCard-230310486212.pdf
B2B Website Audit & Improvement Plan (1).gdoc
B2B Website Audit & Improvement Plan.gdoc
CUETAdmitCard-223510703307.pdf
Calm.ogg
Can

In [11]:
from google.colab import drive
drive.flush_and_unmount()
drive.mount('/content/drive')

Mounted at /content/drive


In [13]:
from google.colab import files
uploaded = files.upload()

Saving EcoSort_TrashNet_split.zip to EcoSort_TrashNet_split.zip


In [14]:
from pathlib import Path
import zipfile

assert len(uploaded) == 1, 'Expected one uploaded ZIP file'
zip_name = next(iter(uploaded))

with zipfile.ZipFile(zip_name) as archive:
    archive.extractall('/content/ecosort_data')

data_dir = Path('/content/ecosort_data/split')
counts = {}

for split in ('train', 'val', 'test'):
    counts[split] = sum(
        f.is_file() and f.suffix.lower() in ('.jpg', '.jpeg', '.png')
        for f in (data_dir / split).rglob('*')
    )
    print(f'{split}: {counts[split]} images')

assert counts == {'train': 1766, 'val': 378, 'test': 383}, 'Unexpected image counts'
print('Dataset ready')


train: 1766 images
val: 378 images
test: 383 images
Dataset ready


In [15]:
import sys
import torch
import torchvision

print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('Torchvision:', torchvision.__version__)
print('GPU available:', torch.cuda.is_available())

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

Python: 3.13.15
PyTorch: 2.11.0+cu130
Torchvision: 0.26.0+cu130
GPU available: True
GPU: Tesla T4


In [16]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

torch.manual_seed(42)

class_names = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

train_dataset = datasets.ImageFolder(data_dir / 'train', train_transform)
val_dataset = datasets.ImageFolder(data_dir / 'val', eval_transform)
test_dataset = datasets.ImageFolder(data_dir / 'test', eval_transform)

for dataset in (train_dataset, val_dataset, test_dataset):
    assert dataset.classes == class_names, 'Class labels do not match'

train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True,
    num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=32, shuffle=False,
    num_workers=2, pin_memory=True
)
test_loader = DataLoader(
    test_dataset, batch_size=32, shuffle=False,
    num_workers=2, pin_memory=True
)

print('Classes:', train_dataset.class_to_idx)
print('Train:', len(train_dataset))
print('Validation:', len(val_dataset))
print('Test:', len(test_dataset))

images, labels = next(iter(train_loader))
print('Image batch shape:', images.shape)
print('Label batch shape:', labels.shape)

Classes: {'cardboard': 0, 'glass': 1, 'metal': 2, 'paper': 3, 'plastic': 4, 'trash': 5}
Train: 1766
Validation: 378
Test: 383
Image batch shape: torch.Size([32, 3, 224, 224])
Label batch shape: torch.Size([32])


In [17]:
from torch import nn
from torchvision.models import resnet18, ResNet18_Weights

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = resnet18(weights=ResNet18_Weights.DEFAULT)

# First, train only the new classification layer.
for parameter in model.parameters():
    parameter.requires_grad = False

model.fc = nn.Linear(model.fc.in_features, len(class_names))
model = model.to(device)

class_counts = torch.bincount(
    torch.tensor(train_dataset.targets),
    minlength=len(class_names)
)
class_weights = len(train_dataset) / (len(class_names) * class_counts.float())

criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
optimizer = torch.optim.Adam(model.fc.parameters(), lr=0.001)

print('Model ready: ResNet18')
print('Device:', device)
print('Output classes:', model.fc.out_features)
print('Training images per class:', class_counts.tolist())

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 91.8MB/s]


Model ready: ResNet18
Device: cuda
Output classes: 6
Training images per class: [282, 350, 287, 415, 337, 95]


In [18]:
best_val_accuracy = -1.0
checkpoint_path = '/content/ecosort_resnet18_best.pth'
history = []

for epoch in range(1, 11):
    # Keep the frozen image-recognition layers unchanged.
    model.eval()
    model.fc.train()
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += labels.size(0)

    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model(images)

            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    train_accuracy = train_correct / train_total
    val_accuracy = val_correct / val_total
    history.append({
        'epoch': epoch,
        'train_accuracy': train_accuracy,
        'val_accuracy': val_accuracy
    })

    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        torch.save({
            'model_state_dict': {
                key: value.detach().cpu()
                for key, value in model.state_dict().items()
            },
            'class_names': class_names,
            'architecture': 'resnet18',
            'image_size': 224,
            'normalize_mean': [0.485, 0.456, 0.406],
            'normalize_std': [0.229, 0.224, 0.225],
            'epoch': epoch,
            'val_accuracy': val_accuracy
        }, checkpoint_path)

    print(
        f'Epoch {epoch}/10 | '
        f'Train: {train_accuracy:.1%} | '
        f'Validation: {val_accuracy:.1%}',
        flush=True
    )

print(f'Finished. Best validation accuracy: {best_val_accuracy:.1%}')
print('Saved:', checkpoint_path)

Epoch 1/10 | Train: 47.1% | Validation: 65.6%
Epoch 2/10 | Train: 67.5% | Validation: 75.1%
Epoch 3/10 | Train: 73.6% | Validation: 74.6%
Epoch 4/10 | Train: 77.4% | Validation: 74.3%
Epoch 5/10 | Train: 77.7% | Validation: 78.0%
Epoch 6/10 | Train: 79.8% | Validation: 81.0%
Epoch 7/10 | Train: 80.9% | Validation: 80.4%
Epoch 8/10 | Train: 81.4% | Validation: 79.9%
Epoch 9/10 | Train: 80.8% | Validation: 80.4%
Epoch 10/10 | Train: 81.2% | Validation: 79.9%
Finished. Best validation accuracy: 81.0%
Saved: /content/ecosort_resnet18_best.pth


In [19]:
from google.colab import files
files.download('/content/ecosort_resnet18_best.pth')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
from sklearn.metrics import accuracy_score, classification_report

checkpoint = torch.load(
    '/content/ecosort_resnet18_best.pth',
    map_location='cpu',
    weights_only=True
)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

actual_labels = []
predicted_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images.to(device))
        predicted_labels.extend(outputs.argmax(1).cpu().tolist())
        actual_labels.extend(labels.tolist())

print('Saved model epoch:', checkpoint['epoch'])
print(f'Test accuracy: {accuracy_score(actual_labels, predicted_labels):.1%}')
print()
print(classification_report(
    actual_labels,
    predicted_labels,
    labels=list(range(len(class_names))),
    target_names=class_names,
    digits=3,
    zero_division=0
))

Saved model epoch: 6
Test accuracy: 78.9%

              precision    recall  f1-score   support

   cardboard      0.927     0.836     0.879        61
       glass      0.714     0.724     0.719        76
       metal      0.713     0.919     0.803        62
       paper      0.854     0.778     0.814        90
     plastic      0.836     0.767     0.800        73
       trash      0.591     0.619     0.605        21

    accuracy                          0.789       383
   macro avg      0.772     0.774     0.770       383
weighted avg      0.797     0.789     0.790       383



In [21]:
from pathlib import Path
from google.colab import files

report = classification_report(
    actual_labels,
    predicted_labels,
    labels=list(range(len(class_names))),
    target_names=class_names,
    digits=3,
    zero_division=0
)

result_path = Path('/content/EcoSort_TrashNet_Test_Results.txt')
result_path.write_text(
    f"Model: ResNet18\n"
    f"Best epoch: {checkpoint['epoch']}\n"
    f"Validation accuracy: {checkpoint['val_accuracy']:.1%}\n"
    f"Test accuracy: {accuracy_score(actual_labels, predicted_labels):.1%}\n\n"
    + report,
    encoding='utf-8'
)

files.download(str(result_path))

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>